In [2]:
#%pip install pandas matplotlib seaborn sqlalchemy ipython-sql==0.4.1 prettytable==2.5.0
#%pip install tabulate  
#%pip install plotly
#%pip install nbformat>=4.2.0

# VDP Analysis
This notebook will be working with the 'VDP_dataset.csv' file, which was extracted from the Dune query at dune.com/queries/7717567. The table will have the following Columns:
 
| Column | Description |
|---|---|
| id | |
| name | |
| website | |
| auth_address | |
| current_commission | |
| total_stake | |
| vdp_stake | |
| Organic_stake | |
| initial_tier | |
| current_tier | |
| dependency_ratio | |
| Rewards_USD | |
| MonthlyReturns_USD | |
| EstimatedMonthlyReturns_USD | |
| underwater_status | |
| graduation_status | |

In [3]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine('sqlite:///vdpdata.db')

csv_file = 'Vdp_dataset.csv'
table_name = 'vdp'
df = pd.read_csv(csv_file)
df.to_sql(table_name, engine, if_exists='replace', index=False)
print('successful')

successful


- How many validators received a delegation?
- Which delegation tier did each validator receive? 
- How concentrated is VDP stake among validators?


In [29]:
import pandas as pd
import plotly.io as pio
import plotly.graph_objects as go
import plotly.express as px

MONAD = {
    "purple":    "#836EF9",
    "blue":      "#200052",
    "berry":     "#A0055D",
    "off_white": "#FBFAF9",
    "black":     "#0E100F",
}
CORTEX_GRAPHITE = "#5B6470"
CORTEX_SLATE    = "#9BA4B0"

COLOR_GROWTH     = MONAD["purple"]
COLOR_BORDERLINE = CORTEX_SLATE
COLOR_SHAME      = MONAD["berry"]
COLOR_NEUTRAL    = CORTEX_GRAPHITE
COLOR_BG         = MONAD["off_white"]
COLOR_TEXT       = MONAD["black"]

CATEGORY_COLORS = {
    "Growing Ecosystem (>=10%)": COLOR_GROWTH,
    "Borderline (5-10%)": COLOR_BORDERLINE,
    "Not Growing (<5%)": COLOR_SHAME,
    "No Data": CORTEX_SLATE,
}

FONT_FAMILY = "Inter, -apple-system, Helvetica Neue, Arial, sans-serif"  
tufte_template = go.layout.Template()
tufte_template.layout = go.Layout(
    font=dict(family=FONT_FAMILY, size=13, color=COLOR_TEXT),
    paper_bgcolor=COLOR_BG,
    plot_bgcolor=COLOR_BG,
    title=dict(font=dict(size=17, color=MONAD["blue"]), x=0.02, xanchor="left"),
    xaxis=dict(showgrid=False, zeroline=False, showline=True,
               linecolor=CORTEX_SLATE, linewidth=1, ticks="outside", tickcolor=CORTEX_SLATE),
    yaxis=dict(showgrid=True, gridcolor="#EDEBF7", gridwidth=1, zeroline=False,
               showline=False, ticks=""),
    legend=dict(bgcolor="rgba(0,0,0,0)", bordercolor="rgba(0,0,0,0)"),
    margin=dict(l=60, r=30, t=60, b=50),
    colorway=[MONAD["purple"], CORTEX_GRAPHITE, MONAD["berry"], CORTEX_SLATE, MONAD["blue"]],
)
pio.templates["tufte_monad"] = tufte_template
pio.templates.default = "tufte_monad"


def categorize_share(share):
    """
    Classifies a validator's organic-stake share of total stake.
    Threshold rationale: a validator needs organic stake to represent a
    meaningful piece of its own total stake to count as having grown the
    ecosystem, rather than sitting on parked/idle VDP capital.
    """
    if pd.isna(share):
        return "No Data"
    if share >= 0.10:
        return "Growing Ecosystem (>=10%)"
    elif share >= 0.05:
        return "Borderline (5-10%)"
    else:
        return "Not Growing (<5%)"


In [30]:
query = """SELECT COUNT(DISTINCT id) as "Total VDP Validators"
FROM 'vdp'
WHERE initial_tier IS NOT NULL"""
df = pd.read_sql(query,engine)
total = df["Total VDP Validators"].iloc[0]
fig = go.Figure(
    go.Indicator(
        mode="number",
        value=total,
        number={
            "font": {
            "color": MONAD["purple"],
            "family": FONT_FAMILY
        }},
        title={"text": "Total VDP Validators",
        "font": {"color": MONAD["blue"]}}
        ))
fig.update_layout(paper_bgcolor=COLOR_BG)
fig.show()

In [31]:
query = """SELECT initial_tier, COUNT(id) as total_delegators
FROM 'vdp'
WHERE initial_tier IS NOT NULL
GROUP BY initial_tier
ORDER BY count(id) desc"""
df = pd.read_sql(query, engine)

# Tufte principle: with more than two categories, an ordered bar is read
# faster and more accurately than pie-slice angles.
df = df.sort_values("total_delegators")
fig = px.bar(
    df, x="total_delegators", y="initial_tier", orientation="h",
    text="total_delegators",
    title="Initial VDP Tier Distribution",
)
fig.update_traces(marker_color=MONAD["purple"], textposition="outside")
fig.update_layout(xaxis_title="Number of Validators", yaxis_title="", showlegend=False)
fig.show()

In [32]:
query = """SELECT current_tier, COUNT(id) as total_delegators
FROM 'vdp'
WHERE current_tier IS NOT NULL
GROUP BY current_tier
ORDER BY count(id) desc"""
df = pd.read_sql(query, engine)

df = df.sort_values("total_delegators")
fig = px.bar(
    df, x="total_delegators", y="current_tier", orientation="h",
    text="total_delegators",
    title="Current VDP Tier Distribution",
)
fig.update_traces(marker_color=MONAD["blue"], textposition="outside")
fig.update_layout(xaxis_title="Number of Validators", yaxis_title="", showlegend=False)
fig.show()

In [7]:
query = """SELECT initial_tier, count(id) as total_delegators
FROM 'vdp'
WHERE initial_tier IS NOT NULL
GROUP BY initial_tier"""
df = pd.read_sql(query, engine)
fig = px.bar(df, x="initial_tier", y="total_delegators", title="VDP Distribution Chart")
#fig.update_layout(, showlegend=False, xaxis_title="Delegation Tier", yaxis="Number of Validators")
fig.show()

Herfindal-Hirschman Index (HHI)

In [34]:
query ="""with shares AS (SELECT vdp_stake * 1.0 / SUM(vdp_stake) OVER() AS share
FROM vdp)
select sum(power(share, 2)) * 10000 AS hhi_10000
FROM shares"""
df = pd.read_sql(query, engine)
df

,hhi_10000
0,54.390096


In [35]:
query="""SELECT dependency_ratio
FROM 'vdp'
"""
df = pd.read_sql(query,engine)
fig = px.histogram(df, x="dependency_ratio", nbins=5, title="Distribution of Validator Dependency Ratio")
fig.update_traces(marker_color=MONAD["purple"], marker_line_color=COLOR_BG, marker_line_width=1)
fig.update_layout(xaxis_title="Dependency Ratio (VDP stake / Total stake)", yaxis_title="Number of Validators")
fig.show()

In [41]:
query = """SELECT name, vdp_stake, organic_stake, total_stake,
                  organic_stake * 1.0 / NULLIF(total_stake, 0) AS organic_share
FROM 'vdp'"""
df = pd.read_sql(query, engine)
df["category"] = df["organic_share"].apply(categorize_share)

# Organic stake is heavily right-skewed -- a handful of validators hold the
# vast majority of it, which crushes everyone else against the origin on a
# linear axis. A log scale spreads the full population out so clusters are
# actually readable. A 1 MON floor is applied purely for display so
# validators with ~0 organic stake still have a plottable position at the
# far left instead of breaking the log axis.
df["organic_stake_plot"] = df["organic_stake"].clip(lower=1)

fig = px.scatter(
    df, x="organic_stake_plot", y="vdp_stake",
    color="category",
    color_discrete_map=CATEGORY_COLORS,
    category_orders={"category": ["Growing Ecosystem (>=10%)", "Borderline (5-10%)", "Not Growing (<5%)", "No Data"]},
    hover_name="name",
    hover_data={"organic_share": ":.1%", "organic_stake_plot": False, "organic_stake": ":,.0f"},
    log_x=True,
    title="VDP Stake vs. Organic Stake, by Ecosystem-Growth Category",
)
fig.update_traces(marker=dict(size=9, opacity=0.8, line=dict(width=0.5, color=COLOR_BG)))

# No inline name labels here -- with dozens of validators clustered by tier,
# on-chart text collides no matter how it's placed. Color + hover carry the
# category signal; the "Naming & Shaming" table further down carries the
# actual names, ranked, without any layout compromise.
fig.add_annotation(
    xref="paper", yref="paper", x=1, y=-0.16, showarrow=False, align="right",
    text="Validator names for the 'Not Growing' group: see the Naming & Shaming table below.",
    font=dict(size=10, color=CORTEX_SLATE),
)

fig.update_layout(
    xaxis_title="Organic Stake (MON, log scale)",
    yaxis_title="VDP Stake (MON)",
    legend_title_text="",
)
fig.show()

In [44]:
query = """SELECT name As "Validator Name", organic_stake As "Organic Stake"
FROM 'vdp'
ORDER BY organic_stake DESC
LIMIT 20"""

df = pd.read_sql(query,engine)
fig = px.bar(df, x="Validator Name", y="Organic Stake", title="Top 20 Validators by Organic Stake")
fig.show()

In [45]:
query = """SELECT name As "Validator Name", Round(organic_stake,2) As "Organic Stake (MON)"
FROM 'vdp'
ORDER BY organic_stake DESC
LIMIT 20"""

df = pd.read_sql(query, engine)
fig = go.Figure(data=[go.Table(
    header=dict(values=list(df.columns), fill_color=MONAD["blue"],
                font=dict(color="white", size=12), align="left", height=32),
    cells=dict(values=[df[c] for c in df.columns],
               fill_color=[[COLOR_BG, "#EFEDFB"] * (len(df)//2 + 1)],
               font=dict(color=COLOR_TEXT, size=11), align="left", height=26),
)])
fig.update_layout(title="Top 20 Validators by Organic Stake", margin=dict(l=10, r=10, t=50, b=10))
fig.show()

In [12]:
query = """SELECT name As "Validator Name", total_stake As "Total Stake"
FROM 'vdp'
ORDER BY total_stake DESC
LIMIT 20"""

df = pd.read_sql(query,engine)
fig = px.bar(df, x="Validator Name", y="Total Stake", title="Top 20 Validators by Total Stake")
fig.show()

In [46]:
query = """
SELECT count(*) as "Total Validators", CASE WHEN graduation_status IS false THEN "undergraduated" else "graduated"
end as "Status"
from 'vdp'
group by 2
"""
df = pd.read_sql(query,engine)
fig = px.pie(df, names="Status", values="Total Validators", title="Validator Graduation Status",
             color="Status",
             color_discrete_map={"graduated": MONAD["purple"], "undergraduated": MONAD["berry"]},
             hole=0.5)
fig.update_traces(
    textposition="outside", textinfo="label+percent",
    marker=dict(line=dict(color=COLOR_BG, width=2)),
)
fig.update_layout(showlegend=False)
fig.show()

In [14]:
query = """
SELECT count(*) as "Total Validators", CASE WHEN underwater_status IS true THEN "Underwater" else "Above Water" 
end as "Status"
from 'vdp'
group by 2
"""
df = pd.read_sql(query,engine)
names = df["Status"]
values = df["Total Validators"]
fig = px.pie(df, names=names, values=values, title="Validator Sustainability")
fig.show()

In [53]:
query = """SELECT name, MonthlyReturns_USD, EstimatedMonthlyReturns_USD,
                  organic_stake * 1.0 / NULLIF(total_stake, 0) AS organic_share
FROM 'vdp'"""
df = pd.read_sql(query, engine)
df = df.rename(columns={
    "MonthlyReturns_USD": "Current Monthly Returns",
    "EstimatedMonthlyReturns_USD": "Estimated Returns Without VDP",
})
df["category"] = df["organic_share"].apply(categorize_share)
df["current_plot"] = df["Current Monthly Returns"].clip(lower=1)
df["estimated_plot"] = df["Estimated Returns Without VDP"].clip(lower=1)

fig = px.scatter(
    df, x="current_plot", y="estimated_plot",
    color="category",
    color_discrete_map=CATEGORY_COLORS,
    category_orders={"category": ["Growing Ecosystem (>=10%)", "Borderline (5-10%)", "Not Growing (<5%)", "No Data"]},
    hover_name="name",
    hover_data={
        "organic_share": ":.1%",
        "current_plot": False, "estimated_plot": False,
        "Current Monthly Returns": ":$,.0f",
        "Estimated Returns Without VDP": ":$,.0f",
    },
    log_x=True, log_y=True,
    title="Current vs. Estimated Returns Without VDP",
)
fig.update_traces(marker=dict(size=9, opacity=0.8, line=dict(width=0.5, color=COLOR_BG)))

# 45-degree reference line: points above the line earn more than their
# organic stake alone would justify -- i.e. VDP is inflating their returns.
max_val = max(df["current_plot"].max(), df["estimated_plot"].max())
min_val = min(df["current_plot"].min(), df["estimated_plot"].min())
fig.add_shape(type="line", x0=min_val, y0=min_val, x1=max_val, y1=max_val,
              line=dict(color=CORTEX_SLATE, width=1, dash="dot"))

fig.update_layout(
    yaxis_tickprefix="$", xaxis_tickprefix="$",
    xaxis_title="Current Monthly Returns (USD)",
    yaxis_title="Estimated Monthly Returns Without VDP (USD)",
    legend_title_text="",
)
fig.show()

In [15]:
query ="""SELECT name AS Validator, ROUND(MonthlyReturns_USD,2) AS "Current Monthly Returns", rOUND(EstimatedMonthlyReturns_USD,2) as "Estimated Returns Without VDP"
FROM 'vdp'
"""
df = pd.read_sql(query, engine)
fig = px.scatter(df, x="Current Monthly Returns", y="Estimated Returns Without VDP", hover_name="Validator")
fig.update_layout(yaxis_tickprefix = '$',
xaxis_tickprefix='$')
fig.show()

In [54]:
query = """
SELECT sum(organic_stake) as stake, 'Organic' as type
from 'vdp'
union
select sum(vdp_stake) as stake, 'VDP' as type
from 'vdp'
"""
df = pd.read_sql(query,engine)
fig = px.pie(df, names="type", values="stake", title="Stake Composition: Organic vs. VDP",
             color="type",
             color_discrete_map={"Organic": MONAD["purple"], "VDP": CORTEX_GRAPHITE},
             hole=0.5)
fig.update_traces(
    textposition="outside", textinfo="label+percent",
    hovertemplate="<b>%{label}</b><br>Stake: %{value:,.0f} MON<br>Share: %{percent}<extra></extra>",
    marker=dict(line=dict(color=COLOR_BG, width=2))
)
fig.update_layout(showlegend=False)
fig.show()

In [55]:
query = """
WITH main AS (
    SELECT row_number() over(order by total_stake desc) as row, name, total_stake, organic_stake
    FROM vdp
    ORDER BY total_stake DESC
    LIMIT 50
),

T20 AS (
    SELECT SUM(total_stake) AS top_20_total, SUM(organic_stake) AS top_20_organic
    FROM main
    WHERE row <= 20
),

T10 AS (
    SELECT SUM(total_stake) AS top_10_total, SUM(organic_stake) AS top_10_organic
    FROM main
    WHERE row <= 10
),

T50 AS (
    SELECT SUM(total_stake) AS top_50_total, SUM(organic_stake) AS top_50_organic
    FROM main
    WHERE row <= 50
), 

totals AS (
    SELECT sum(total_stake) AS total, sum(organic_stake) AS organic
    FROM vdp
)

SELECT 'Top 10 Share' AS metric, 
       top_10_total / total AS Current, 
       top_10_organic / organic AS "Without VDP"
FROM totals, T10

UNION ALL

SELECT 'Top 20 Share' AS metric, 
       top_20_total / total AS Current, 
       top_20_organic / organic AS "Without VDP"
FROM totals, T20

UNION ALL

SELECT 'Top 50 Share' AS metric, 
       top_50_total / total AS Current, 
       top_50_organic / organic AS "Without VDP"
FROM totals, T50
"""

df = pd.read_sql(query, engine)
df['Current'] = df['Current'].map('{:.2%}'.format)
df['Without VDP'] = df['Without VDP'].map('{:.2%}'.format)

fig = go.Figure(
    data = [go.Table(
        header=dict(values=list(df.columns),
        align='left'),
        cells=dict(values=[df[col] for col in df.columns],
        align='left')
    )]
)
fig.show()

In [57]:
query = """
WITH main AS (
    SELECT row_number() over(order by total_stake desc) as row, name, total_stake, organic_stake
    FROM vdp
    ORDER BY total_stake DESC
    LIMIT 50
),

T20 AS (
    SELECT SUM(total_stake) AS top_20_total, SUM(organic_stake) AS top_20_organic
    FROM main
    WHERE row <= 21
    and lower(name) is not 'backpack'
),

T10 AS (
    SELECT SUM(total_stake) AS top_10_total, SUM(organic_stake) AS top_10_organic
    FROM main
    WHERE row <= 11
    and lower(name) is not 'backpack'
),

T50 AS (
    SELECT SUM(total_stake) AS top_50_total, SUM(organic_stake) AS top_50_organic
    FROM main
    WHERE row <= 51
    and lower(name) is not 'backpack'
), 

totals AS (
    SELECT sum(total_stake) AS total, sum(organic_stake) AS organic
    FROM vdp
    where lower(name) is not 'backpack'
)

SELECT 'Top 10 Share' AS metric, 
       top_10_total / total AS Current, 
       top_10_organic / organic AS "Without VDP"
FROM totals, T10

UNION ALL

SELECT 'Top 20 Share' AS metric, 
       top_20_total / total AS Current, 
       top_20_organic / organic AS "Without VDP"
FROM totals, T20

UNION ALL

SELECT 'Top 50 Share' AS metric, 
       top_50_total / total AS Current, 
       top_50_organic / organic AS "Without VDP"
FROM totals, T50
"""

df = pd.read_sql(query, engine)
df['Current'] = df['Current'].map('{:.2%}'.format)
df['Without VDP'] = df['Without VDP'].map('{:.2%}'.format)

fig = go.Figure(data=[go.Table(
    header=dict(values=list(df.columns), fill_color=MONAD["blue"],
                font=dict(color="white", size=12), align="left", height=32),
    cells=dict(values=[df[c] for c in df.columns],
               fill_color=[[COLOR_BG, "#EFEDFB"] * (len(df)//2 + 1)],
               font=dict(color=COLOR_TEXT, size=11), align="left", height=28),
)])
fig.update_layout(title="Stake Concentration Excluding Backpack: Current vs. Without VDP", margin=dict(l=10, r=10, t=50, b=10))
fig.show()

In [58]:
query = """WITH tot AS (select sum(total_stake) as total, sum(organic_stake) as organic
from 'vdp')

SELECT total_stake/total as "Total Backpack Concentration", organic_stake/organic as "Total Organic Concentration"
from 'vdp', 'tot' 
where lower(name) is 'backpack'

"""
df = pd.read_sql(query, engine)
df['Total Backpack Concentration'] = df['Total Backpack Concentration'].map('{:.2%}'.format)
df['Total Organic Concentration'] = df['Total Organic Concentration'].map('{:.2%}'.format)

fig = go.Figure(data=[go.Table(
    header=dict(values=list(df.columns), fill_color=MONAD["berry"],
                font=dict(color="white", size=12), align="left", height=32),
    cells=dict(values=[df[c] for c in df.columns],
               fill_color=[[COLOR_BG]], font=dict(color=COLOR_TEXT, size=11), align="left", height=28),
)])
fig.update_layout(title="Backpack Concentration", margin=dict(l=10, r=10, t=50, b=10))
fig.show()

In [59]:
query = """select name, organic_stake/total_stake as share
from 'vdp'
"""

df = pd.read_sql(query, engine)
fig = px.ecdf(df,
              x='share',
              title="Distribution of Organic Stake Share")
fig.update_traces(line_color=MONAD["purple"], line_width=2.5)

for x in [0.05, 0.10, 0.25, 0.50]:
    fig.add_vline(
        x=x,
        line_dash="dash",
        line_color=MONAD["berry"] if x <= 0.05 else CORTEX_SLATE,
        annotation_text=f"{int(x*100)}%",
        annotation_position="top"
    )
fig.update_layout(
    xaxis_title="Organic Stake Share",
    yaxis_title="Cumulative Share of Validators"
)

fig.update_xaxes(tickformat='.0%')
fig.update_yaxes(tickformat='.0%')

fig.update_traces(hovertemplate=
                  "<b>Organic Stake Share:</b> %{x:.1%}<br>"
                  "<b>Validators Below This Level:</b> %{y:.1%}"
                  "<extra></extra>")
fig.show()

In [60]:
query = """
SELECT name,
       organic_stake, vdp_stake, total_stake,
       organic_stake * 1.0 / NULLIF(total_stake, 0) AS organic_share,
       organic_stake * 1.0 / NULLIF(vdp_stake, 0) AS organic_per_vdp
FROM 'vdp'
"""
df = pd.read_sql(query, engine)
df["category"] = df["organic_share"].apply(categorize_share)

worst = df.nsmallest(15, "organic_per_vdp").sort_values("organic_per_vdp")

fig = px.bar(
    worst, x="organic_per_vdp", y="name", orientation="h",
    color="category",
    color_discrete_map=CATEGORY_COLORS,
    title="Lowest Capital Efficiency: Organic Stake Generated per Unit of VDP Stake",
    text=worst["organic_per_vdp"].map("{:.2f}".format),
)
fig.update_traces(textposition="outside")
fig.update_layout(xaxis_title="Organic MON per VDP MON", yaxis_title="", legend_title_text="")
fig.show()

In [61]:
query = """
SELECT name,
       total_stake, organic_stake, vdp_stake,
       organic_stake * 1.0 / NULLIF(total_stake, 0) AS organic_share,
       MonthlyReturns_USD, EstimatedMonthlyReturns_USD,
       dependency_ratio
FROM 'vdp'
"""
df = pd.read_sql(query, engine)
df["returns_gap_usd"] = df["MonthlyReturns_USD"] - df["EstimatedMonthlyReturns_USD"]

# Naming & Shaming: validators below the 5% organic-share threshold,
# ranked by how much VDP-subsidized return they draw while contributing
# least to organic growth.
shame_list = (
    df[df["organic_share"] < 0.05]
    .sort_values("returns_gap_usd", ascending=False)
    [["name", "organic_share", "total_stake", "returns_gap_usd", "dependency_ratio"]]
    .copy()
)
shame_list["organic_share"] = shame_list["organic_share"].map("{:.1%}".format)
shame_list["total_stake"] = shame_list["total_stake"].map("{:,.0f}".format)
shame_list["returns_gap_usd"] = shame_list["returns_gap_usd"].map("${:,.0f}".format)
shame_list["dependency_ratio"] = shame_list["dependency_ratio"].map("{:.2f}".format)
shame_list.columns = ["Validator", "Organic Share", "Total Stake (MON)", "VDP Return Gap (USD/mo)", "Dependency Ratio"]

row_h = 26
fig = go.Figure(data=[go.Table(
    header=dict(values=list(shame_list.columns), fill_color=MONAD["berry"],
                font=dict(color="white", size=12), align="left", height=32),
    cells=dict(values=[shame_list[c] for c in shame_list.columns],
               fill_color=[[COLOR_BG, "#F7EAF1"] * (len(shame_list)//2 + 1)],
               font=dict(color=COLOR_TEXT, size=11), align="left", height=row_h),
)])
fig.update_layout(
    title=f"Validators Not Growing the Ecosystem (Organic Share Below 5%) -- n={len(shame_list)}",
    margin=dict(l=10, r=10, t=50, b=10),
)
fig.show()